# 00 — META PIPELINE

Runs every pipeline notebook in sequence (01 → 24), each in its own isolated kernel.
Outputs are saved to `run/outputs/` so originals are never modified.

| # | Notebook | Description |
|---|----------|-------------|
| 01 | `01_corpus_collection` | Collect ARO source corpus |
| 02 | `02_knowledge_extraction` | Extract structured knowledge |
| 03 | `03_llm_knowledge_extraction` | LLM-assisted knowledge extraction |
| 04 | `04_warmstart_finetune` | Warm-start fine-tune |
| 05 | `05_actions_training` | Actions training data |
| 06 | `06_repl_execution_training` | REPL execution training data |
| 07 | `07_book_qa_pairs` | Book Q&A pairs |
| 08 | `08_wiki_training` | Wiki training data |
| 09 | `09_git_training` | Git training data |
| 10 | `10_synthetic_data_generation` | Synthetic data generation |
| 11 | `11_function_calling` | Function calling training |
| 12 | `12_application_prompts` | Application prompt training data |
| 13 | `13_examples_repo_training` | Examples repository training data |
| 14 | `14_external_repo_training` | External repository training data |
| 15 | `15_comment_extraction` | Comment extraction training data |
| 16 | `16_validation` | Validate all training pairs |
| 17 | `17_dataset_assembly` | Assemble final dataset |
| 18 | `18_finetune` | Full fine-tune (SFT) on 30B |
| 19 | `19_preference_sft` | Preference-filtered SFT |
| 20 | `20_evaluation` | Evaluate model |
| 21 | `21_iterative_loop` | Iterative self-improvement loop |
| 22 | `23_distillation` | Distill 30B → Qwen3-8B student |
| 23 | `24_package` | Package & distribute |

**Excluded from the automated run** (manual testing only — both require interactive input):
- `22_chat_teacher` — chat with the teacher model before distillation
- `25_chat` — `aro ask` REPL against the packaged student

In [1]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'nbconvert', 'nbclient', 'nbformat'], check=False)


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['pip', 'install', '-q', 'nbconvert', 'nbclient', 'nbformat'], returncode=0)

In [2]:
import json
import time
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from IPython.display import display, clear_output, HTML

SCRIPT_DIR  = Path('.').resolve()
OUTPUT_DIR  = SCRIPT_DIR / 'run' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Notebooks to run in order ──────────────────────────────────────────────
# NB22 (chat_teacher) and NB25 (aro ask chat REPL) are deliberately excluded —
# both require interactive stdin and are for manual testing only.
NOTEBOOKS = [
    ('01',  '01_corpus_collection',          'Collect ARO source corpus'),
    ('02',  '02_knowledge_extraction',       'Extract structured knowledge'),
    ('03',  '03_llm_knowledge_extraction',   'LLM-assisted knowledge extraction'),
    ('04',  '04_warmstart_finetune',         'Warm-start fine-tune'),
    ('05',  '05_actions_training',           'Actions training data'),
    ('06',  '06_repl_execution_training',    'REPL execution training data'),
    ('07',  '07_book_qa_pairs',              'Book Q&A pairs'),
    ('08',  '08_wiki_training',              'Wiki training data'),
    ('09',  '09_git_training',               'Git training data'),
    ('10',  '10_synthetic_data_generation',  'Synthetic data generation'),
    ('11',  '11_function_calling',           'Function calling training'),
    ('12',  '12_application_prompts',        'Application prompt training data'),
    ('13',  '13_examples_repo_training',     'Examples repository training data'),
    ('14',  '14_external_repo_training',     'External repository training data'),
    ('15',  '15_comment_extraction',         'Comment extraction training data'),
    ('16',  '16_validation',                 'Validate all training pairs'),
    ('17',  '17_dataset_assembly',           'Assemble final dataset'),
    ('18',  '18_finetune',                   'Full fine-tune (SFT)'),
    ('19',  '19_preference_sft',             'Preference-filtered SFT'),
    ('20',  '20_evaluation',                 'Evaluate model'),
    ('21',  '21_iterative_loop',             'Iterative self-improvement loop'),
    ('23',  '23_distillation',               'Distill 30B → Qwen3-8B student'),
    ('24',  '24_package',                    'Package & distribute'),
]

# Cell execution timeout per notebook (seconds).
# Use 0 to disable timeout entirely (run until completion).
DEFAULT_TIMEOUT = 7200   # 2 h
TIMEOUT_OVERRIDES = {
    '04':  14400,   # warm-start fine-tune
    '07':  14400,   # book Q&A: many LLM calls
    '08':  14400,   # wiki training: LLM calls
    '09':  10800,   # git training: commit mining + LLM
    '10':  0,       # synthetic data: no timeout — runs until done
    '11':  10800,   # function calling: many LLM calls
    '12':  10800,   # application prompts: LLM paraphrases
    '13':  10800,   # examples repo: clone + LLM pairs
    '14':  10800,   # external repos: clone + LLM pairs
    '15':  10800,   # comment extraction: LLM pairs
    '18':  0,       # full fine-tune: no timeout
    '19':  14400,   # preference SFT: generation + training
    '21':  0,       # iterative loop: no timeout
    '23':  0,       # distillation: no timeout
}

# ── Kernel detection & registration ───────────────────────────────────────
def _list_kernels():
    """Return dict of available kernel names → spec paths."""
    try:
        out = subprocess.check_output(
            [sys.executable, '-m', 'jupyter', 'kernelspec', 'list', '--json'],
            text=True, stderr=subprocess.DEVNULL, timeout=30,
        )
        return json.loads(out).get('kernelspecs', {})
    except Exception:
        return {}


def _register_kernel(name='aro-python3'):
    """Register sys.executable as a Jupyter kernel."""
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'ipykernel', 'install',
             '--user', '--name', name, '--display-name', f'Python 3 ({name})'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=60,
        )
        return name
    except Exception as exc:
        print(f'  WARNING: could not register kernel "{name}": {exc}')
        return None


_PYTHON_KERNEL_NAMES = ('python3', 'python', 'aro-python3', 'Python3', 'Python 3')

specs = _list_kernels()
KERNEL_NAME = next(
    (k for k in _PYTHON_KERNEL_NAMES if k in specs),
    None,
)

if KERNEL_NAME is None:
    print(f'No Python kernel found in: {list(specs.keys()) or ["(none)"]}')
    print('Registering current Python as "aro-python3"...')
    KERNEL_NAME = _register_kernel('aro-python3')
    if KERNEL_NAME:
        print(f'  Registered: {KERNEL_NAME}  ({sys.executable})')
    else:
        print('  FAILED — check that ipykernel is installed:')
        print(f'    {sys.executable} -m pip install ipykernel')
else:
    print(f'Kernel: {KERNEL_NAME}  ({specs[KERNEL_NAME].get("spec",{}).get("argv",["?"])[0]})')

print(f'Output dir: {OUTPUT_DIR}')
print(f'Notebooks : {len(NOTEBOOKS)}')

Kernel: python3  (python)
Output dir: /Users/kris/Projects/ARO/ARO-Train/Train/script/run/outputs
Notebooks : 23


In [3]:
# ── Status table renderer ──────────────────────────────────────────────────

STATUS_ICON = {
    'pending':  '⏳',
    'running':  '🔄',
    'done':     '✅',
    'failed':   '❌',
    'skipped':  '⏭️',
}

def _fmt_duration(seconds):
    if seconds is None:
        return '—'
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h:
        return f'{h}h {m:02d}m {s:02d}s'
    if m:
        return f'{m}m {s:02d}s'
    return f'{s}s'


def render_status(rows, overall_start):
    """Render a coloured HTML status table and update the cell output in place."""
    elapsed = time.time() - overall_start
    done_count  = sum(1 for r in rows if r['status'] == 'done')
    fail_count  = sum(1 for r in rows if r['status'] == 'failed')
    total = len(rows)

    header = (
        f'<h3 style="margin:0 0 8px">🚀 META PIPELINE &nbsp;'
        f'<span style="font-size:0.75em;color:#888">'
        f'{done_count}/{total} done · {fail_count} failed · elapsed {_fmt_duration(elapsed)}'
        f'</span></h3>'
    )

    row_html = ''
    for r in rows:
        icon    = STATUS_ICON.get(r['status'], '?')
        dur     = _fmt_duration(r.get('duration'))
        err_td  = f'<td style="color:#c00;font-size:0.8em">{r.get("error","")}</td>'
        bg      = {
            'running': '#fffbe6',
            'done':    '#f0fff4',
            'failed':  '#fff0f0',
        }.get(r['status'], '')
        bg_attr = f'style="background:{bg}"' if bg else ''
        row_html += (
            f'<tr {bg_attr}>'
            f'<td style="text-align:center">{icon}</td>'
            f'<td><code>{r["num"]}</code></td>'
            f'<td>{r["name"]}</td>'
            f'<td style="color:#555">{r["description"]}</td>'
            f'<td style="text-align:right">{dur}</td>'
            f'{err_td}'
            f'</tr>'
        )

    table = (
        '<table style="border-collapse:collapse;width:100%;font-size:0.9em">'
        '<thead><tr>'
        '<th></th><th>#</th><th>Notebook</th><th>Description</th>'
        '<th style="text-align:right">Duration</th><th>Error</th>'
        '</tr></thead>'
        f'<tbody>{row_html}</tbody>'
        '</table>'
    )
    clear_output(wait=True)
    display(HTML(header + table))


print('Helpers ready.')

Helpers ready.


In [4]:
# ── Per-notebook executor ──────────────────────────────────────────────────

def run_notebook(num, name, timeout, kernel_name):
    """
    Execute `name.ipynb` via `jupyter nbconvert --execute`.
    Saves the executed copy to OUTPUT_DIR / `name.ipynb`.
    A timeout of 0 means no timeout (run until completion).
    Returns (success: bool, error_msg: str | None).
    """
    src  = SCRIPT_DIR / f'{name}.ipynb'
    dest = OUTPUT_DIR / f'{name}.ipynb'

    if not src.exists():
        return False, f'file not found: {src.name}'

    if kernel_name is None:
        return False, 'no Python kernel available — run setup cell first'

    # timeout=0 → no limit: use -1 for nbconvert (disables cell timeout)
    # and None for subprocess (no outer guard)
    nb_timeout  = str(timeout) if timeout > 0 else '-1'
    sub_timeout = (timeout + 120) if timeout > 0 else None

    cmd = [
        sys.executable, '-m', 'jupyter', 'nbconvert',
        '--to',          'notebook',
        '--execute',
        '--output',      str(dest),
        '--ExecutePreprocessor.timeout',     nb_timeout,
        '--ExecutePreprocessor.kernel_name', kernel_name,
        str(src),
    ]

    log_file = OUTPUT_DIR / f'{name}.log'
    try:
        with open(log_file, 'w') as log:
            result = subprocess.run(
                cmd,
                stdout=log,
                stderr=subprocess.STDOUT,
                timeout=sub_timeout,
            )
        if result.returncode != 0:
            # Pull the last meaningful line from the log as the error summary
            lines = log_file.read_text().splitlines()
            snippet = next(
                (l.strip() for l in reversed(lines) if l.strip() and not l.startswith('[')),
                'non-zero exit'
            )[:120]
            return False, snippet
        return True, None
    except subprocess.TimeoutExpired:
        return False, f'timed out after {_fmt_duration(timeout)}'
    except Exception as exc:
        return False, str(exc)[:120]


print('Runner ready.')

Runner ready.


## ▶ Run the pipeline

Execute the cell below to start.  
The status table updates live as each notebook finishes.

> **Tip:** set `SKIP` to a set of notebook numbers (e.g. `{'04', '15'}`) to skip specific steps.

In [ ]:
# Numbers to skip — set to empty set to run everything
SKIP = set()
STOP_ON_FAILURE = True    # halt the pipeline when a notebook fails

if KERNEL_NAME is None:
    raise RuntimeError(
        'No Python kernel is available. '
        'Re-run the setup cell above — it will attempt to register one automatically.'
    )

# ── Build initial status rows ──────────────────────────────────────────────
rows = [
    {'num': num, 'name': name, 'description': desc,
     'status': 'skipped' if num in SKIP else 'pending',
     'duration': None, 'error': ''}
    for num, name, desc in NOTEBOOKS
]

pipeline_start = time.time()
render_status(rows, pipeline_start)

# ── Execute ────────────────────────────────────────────────────────────────
failed_any = False

for row in rows:
    if row['status'] == 'skipped':
        continue

    row['status'] = 'running'
    render_status(rows, pipeline_start)
    print(f'▶ [{row["num"]}/{len(rows):02d}]  {row["name"]}  —  {row["description"]}',
          flush=True)

    timeout = TIMEOUT_OVERRIDES.get(row['num'], DEFAULT_TIMEOUT)
    t0 = time.time()
    success, error = run_notebook(row['num'], row['name'], timeout, KERNEL_NAME)
    row['duration'] = time.time() - t0

    if success:
        row['status'] = 'done'
        print(f'  ✅ done     {row["name"]}  ({_fmt_duration(row["duration"])})',
              flush=True)
    else:
        row['status']  = 'failed'
        row['error']   = error or ''
        failed_any     = True
        print(f'  ❌ FAILED   {row["name"]}  ({_fmt_duration(row["duration"])})',
              flush=True)
        print(f'     error: {row["error"]}', flush=True)
        render_status(rows, pipeline_start)
        if STOP_ON_FAILURE:
            # Mark remaining notebooks as skipped
            idx = rows.index(row)
            for r in rows[idx + 1:]:
                if r['status'] == 'pending':
                    r['status'] = 'skipped'
                    r['error']  = 'skipped (pipeline halted on failure)'
            break

    render_status(rows, pipeline_start)

# ── Final summary ──────────────────────────────────────────────────────────
total_elapsed = time.time() - pipeline_start
done_count    = sum(1 for r in rows if r['status'] == 'done')
fail_count    = sum(1 for r in rows if r['status'] == 'failed')
skip_count    = sum(1 for r in rows if r['status'] == 'skipped')

print(f'\nPipeline finished in {_fmt_duration(total_elapsed)}')
print(f'  ✅ done:    {done_count}')
print(f'  ❌ failed:  {fail_count}')
print(f'  ⏭️  skipped: {skip_count}')
print(f'\nKernel used:  {KERNEL_NAME}')
print(f'Outputs:      {OUTPUT_DIR}')

if failed_any:
    print('\nFailed notebooks:')
    for r in rows:
        if r['status'] == 'failed':
            log = OUTPUT_DIR / f'{r["name"]}.log'
            print(f'  {r["num"]} {r["name"]}')
            print(f'     error: {r["error"]}')
            print(f'     log:   {log}')

,#,Notebook,Description,Duration,Error
✅,01,01_corpus_collection,Collect ARO source corpus,1s,
✅,02,02_knowledge_extraction,Extract structured knowledge,1s,
🔄,03,03_llm_knowledge_extraction,LLM-assisted knowledge extraction,—,
⏳,04,04_warmstart_finetune,Warm-start fine-tune,—,
⏳,05,05_actions_training,Actions training data,—,
⏳,06,06_repl_execution_training,REPL execution training data,—,
⏳,07,07_book_qa_pairs,Book Q&A pairs,—,
⏳,08,08_wiki_training,Wiki training data,—,
⏳,09,09_git_training,Git training data,—,
⏳,10,10_synthetic_data_generation,Synthetic data generation,—,


▶ [03/23]  03_llm_knowledge_extraction  —  LLM-assisted knowledge extraction


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9', 'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '00_META_PIPELINE.png'

_done   = [r for r in rows if r['status'] == 'done']
_failed = [r for r in rows if r['status'] == 'failed']
_skip   = [r for r in rows if r['status'] == 'skipped']
_run    = [r for r in rows if r['status'] not in ('skipped',)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: duration per notebook (horizontal bar) ─────────────────────
_labels = [f'NB{int(r["num"]):02d}' for r in _run]
_durs   = [r.get('duration') or 0 for r in _run]
_colors = ['#2ecc71' if r['status'] == 'done' else '#e74c3c' for r in _run]
y = range(len(_labels))
ax1.barh(y, [d / 60 for d in _durs], color=_colors, edgecolor='white', height=0.6)
ax1.set_yticks(list(y))
ax1.set_yticklabels(_labels, fontsize=8)
ax1.set_xlabel('Minutes')
ax1.set_title('Duration per Notebook', fontsize=12, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# ── Right: status donut ──────────────────────────────────────────────
_counts = [len(_done), len(_failed), len(_skip)]
_slabels = [f'Done ({len(_done)})', f'Failed ({len(_failed)})', f'Skipped ({len(_skip)})']
_scolors = ['#2ecc71', '#e74c3c', '#bdc3c7']
_nz = [(c, l, cl) for c, l, cl in zip(_counts, _slabels, _scolors) if c > 0]
if _nz:
    c2, l2, cl2 = zip(*_nz)
    ax2.pie(c2, labels=l2, colors=cl2, autopct='%1.0f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    ax2.add_artist(plt.Circle((0, 0), 0.5, fc='white'))
    _total_m = sum(_durs) / 60
    ax2.text(0, 0, f'{_total_m:.0f}m\ntotal', ha='center', va='center',
             fontsize=13, fontweight='bold')
ax2.set_title('Pipeline Status', fontsize=12, fontweight='bold')

fig.suptitle('Meta Pipeline — Execution Summary', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')
